IMPORTING MODULES

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
import numpy as np
import scipy.io as sio
import mne
from mne.time_frequency import psd_array_welch
from antropy import hjorth_params, spectral_entropy, perm_entropy
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from imblearn.over_sampling import SMOTE


Preprocessing and feature extraction

In [ ]:
sfreq = 256
mne.set_log_level('ERROR')

BANDS = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta":  (13, 30),
    "gamma": (30, 45)
}


In [ ]:
def extract_features(epoch_data, sf):
    n_channels = epoch_data.shape[0]
    nperseg = min(256, epoch_data.shape[1])
    feats = np.zeros((n_channels, 10))
    for ch in range(n_channels):
        x = epoch_data[ch]
        psd, freqs = psd_array_welch(x, sf, n_fft=nperseg)
        band_powers = [np.mean(psd[(freqs >= low) & (freqs <= high)]) for (low, high) in BANDS.values()]
        hjorth_mob, hjorth_comp = hjorth_params(x)
        hjorth_act = np.var(x)
        spec_ent = spectral_entropy(x, sf)
        perm_ent_val = perm_entropy(x, normalize=True)
        feats[ch, :] = np.hstack([band_powers, [hjorth_act, hjorth_mob, hjorth_comp, spec_ent, perm_ent_val]])
    return feats.flatten()

In [ ]:
def process_folder(path):
    X, y = [], []
    info = None
    max_windows = 0

    files = [
        os.path.join(root, f)
        for root, _, filenames in os.walk(path)
        for f in filenames if f.endswith(".mat") and "E0" not in f
    ]

    for file in tqdm(files, desc="Processing"):
        try:
            mat = sio.loadmat(file)
            data = mat.get('Data')
            if data is None:
                continue

            if info is None:
                ch_names = [f"EEG{i}" for i in range(data.shape[0])]
                info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=['eeg'] * len(ch_names))

            raw = mne.io.RawArray(data, info, verbose='ERROR')
            raw.filter(1., 40., verbose='ERROR')
            epochs = mne.make_fixed_length_epochs(raw, duration=2.0, overlap=0.5, verbose='ERROR')

            X_data = epochs.get_data()
            num_windows = X_data.shape[0]
            max_windows = max(max_windows, num_windows)

            X_features = np.array([extract_features(epoch, sfreq) for epoch in X_data])
            X.append(X_features)
            if "E1" in file or "E2" in file:
                label = 0
            elif "E3" in file:
                label = 1
            elif "E4" in file or "E5" in file:
                label = 2
            else:
                continue

            y.append(np.full(len(X_features), label))
        except Exception:
            continue

    X_final = np.concatenate(X, axis=0)
    y_final = np.concatenate(y, axis=0)
      mean_features = np.mean(X_final, axis=0)
  std_features = np.std(X_final, axis=0)
  max_features = np.max(X_final, axis=0)
  np.save("X_final.npy", X_final)
  np.save("y_final.npy", y_final)
  np.save("mean_features.npy", mean_features)
  np.save("std_features.npy", std_features)
  np.save("max_features.npy", max_features)
  np.save("max_windows.npy", np.array([max_windows]))
  return X_final, y_final, mean_features, std_features, max_features, max_windows


In [ ]:
if __name__ == "__main__":
    path = "/content/drive/MyDrive/DEED"
    process_folder(path)

In [ ]:

X_final = X_final.astype(np.float32)
y_final = y_final.astype(np.int64)
X_final = X_final.reshape(X_final.shape[0], 6, 10)


LSTM MODEL WITH TIME2VEC

In [ ]:
class Time2Vec(nn.Module):
    def __init__(self, d_model=128):
        super().__init__()
        self.d_model = d_model
        self.w0 = nn.Parameter(torch.randn(1, 1))
        self.b0 = nn.Parameter(torch.randn(1, 1))
        self.w = nn.Parameter(torch.randn(1, d_model - 1))
        self.b = nn.Parameter(torch.randn(1, d_model - 1))

    def forward(self, t):
        linear_term = self.w0 * t + self.b0
        periodic_terms = torch.sin(self.w * t + self.b)
        return torch.cat([linear_term, periodic_terms], dim=-1)

In [ ]:
class EEGLSTM(nn.Module):
    def __init__(self, feature_dim, d_model=128, hidden_dim=256,
                 num_layers=2, drop_prob=0.3, num_classes=3, use_time2vec=True):
        super().__init__()
        self.use_time2vec = use_time2vec
        self.input_proj = nn.Linear(feature_dim, d_model)
        if use_time2vec:
            self.time2vec = Time2Vec(d_model=d_model)
        self.lstm = nn.LSTM(input_size=d_model, hidden_size=hidden_dim,
                            num_layers=num_layers, batch_first=True,
                            dropout=drop_prob, bidirectional=False)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        x_proj = self.input_proj(x)
        if self.use_time2vec:
            t = torch.arange(seq_len, device=x.device).unsqueeze(0).unsqueeze(-1).repeat(batch_size, 1, 1).float()
            t2v = self.time2vec(t)
            x_proj = x_proj + t2v
        lstm_out, _ = self.lstm(x_proj)
        out = lstm_out[:, -1, :]
        logits = self.classifier(out)
        return logits

Variables, smote and train-test split

In [ ]:
n_samples, seq_len, feature_dim = X_final.shape
X_flat = X_final.reshape(n_samples, -1)
y = y_final
smote = SMOTE(sampling_strategy='auto', random_state=42)
X_res, y_res = smote.fit_resample(X_flat, y)
X_res = X_res.reshape(-1, seq_len, feature_dim)
X_tensor = torch.tensor(X_res, dtype=torch.float32)
y_tensor = torch.tensor(y_res, dtype=torch.long)
dataset = TensorDataset(X_tensor, y_tensor)

In [ ]:
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

Model training and evaluation

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EEGLSTM(feature_dim=feature_dim).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
num_epochs = 150
for epoch in range(num_epochs):
    model.train()
    total_loss, total_acc = 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        preds = torch.argmax(logits, dim=-1)
        acc = (preds == y_batch).float().mean()
        total_loss += loss.item()
        total_acc += acc.item()
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {total_loss/len(train_loader):.4f}, Train Acc: {total_acc/len(train_loader):.4f}")
model.eval()


In [ ]:
test_acc = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        logits = model(X_batch)
        preds = torch.argmax(logits, dim=-1)
        test_acc += (preds == y_batch).float().sum().item()
test_acc /= len(test_dataset)
print(f"Test Accuracy: {test_acc:.4f}")

Saving model

In [ ]:
torch.save(model.state_dict(), "dream.pth")